# CS 195: Natural Language Processing

## Semantic Similarity With WordNet
Over the past few weeks, we've been talking about using *embeddings* to represent the meanings of words as a vector. This allows us to (among other things) compare the meanings of words. 

![Visualizing word embeddings](./images/embedding-vis.png)

image source: [https://stackoverflow.com/questions/46155868/keras-embedding-layer](https://stackoverflow.com/questions/46155868/keras-embedding-layer)

There are other approaches to comparing word meanings, though (as well as other NLP tasks). Before neural networks could learn word meaning, linguists created tools to assist in natural language processing. Today, we'll be looking at using one of those tools - **WordNet** - to compare word meanings. We call this *semantic similarity*. 

## References
* WordNet Homepage: [https://wordnet.princeton.edu/](https://wordnet.princeton.edu/)
* Wordnet on Wikipedia: [https://en.wikipedia.org/wiki/WordNet](https://en.wikipedia.org/wiki/WordNet)
* WordNet sample usage with Python: [https://www.nltk.org/howto/wordnet.html](https://www.nltk.org/howto/wordnet.html)
* WordNet glossary of terms: [https://wordnet.princeton.edu/documentation/wngloss7wn](https://wordnet.princeton.edu/documentation/wngloss7wn)
* Intro to WordNet with Python NLTK: [https://medium.com/@don_khozzy/dive-into-wordnet-with-nltk-b313c480e788](https://medium.com/@don_khozzy/dive-into-wordnet-with-nltk-b313c480e788)


In [2]:
import sys
!{sys.executable} -m pip install nltk

## WordNet?
WordNet is a *lexical database* of English words. It stores words, definitions, and different kinds of relationships between them for 155,000 English words. For now, think about it like a dictionary & thesaurus; it's more than that, but let's start by looking up what `lexical` means. 

In [1]:
import nltk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to /home/evan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
print("Lexical means:")
wn.synsets("lexical")[0].definition()

Lexical means:


'of or relating to words'

## Synsets
What's actually going on here?  

WordNet is a network of words, and its core data structure is the *synset* (synonym set). "Synonym" here refers to a [cognitive synonym](https://en.wikipedia.org/wiki/Cognitive_synonymy), which means that **each synset is centered around a single concept or definition**. We just looked at the first synset that `lexical` appears in, and got the definition associated with it. 

To illustrate this concept of a synset, let's take a look at some furniture.  

In [3]:
# sofa
sofa_synset = wn.synsets("sofa")[0]
print("sofa:")
print(sofa_synset.definition())
print(sofa_synset, '\n') # we'll cover what this notation means in a few minutes

# couch
couch_synset = wn.synsets("couch")[0]
print("couch:")
print(couch_synset.definition())
print(couch_synset)

sofa:
an upholstered seat for more than one person
Synset('sofa.n.01') 

couch:
an upholstered seat for more than one person
Synset('sofa.n.01')


These words have identical definitions - they're both in the same synset (the one associated with "an upholstered seat for more than one person").  
```txt
 _________________________
{        Synset           }
{  "upholstered seat..."  }
{    - couch              }
{    - sofa               }
{    - ...                }
{_________________________}
```

Because words often have multiple meanings, they're often in multiple synsets. Let's look at all the synsets for the word "bank".

In [4]:
bank_synsets = wn.synsets("bank")
print(bank_synsets)
print("bank has", len(bank_synsets), "definitions")

[Synset('bank.n.01'), Synset('depository_financial_institution.n.01'), Synset('bank.n.03'), Synset('bank.n.04'), Synset('bank.n.05'), Synset('bank.n.06'), Synset('bank.n.07'), Synset('savings_bank.n.02'), Synset('bank.n.09'), Synset('bank.n.10'), Synset('bank.v.01'), Synset('bank.v.02'), Synset('bank.v.03'), Synset('bank.v.04'), Synset('bank.v.05'), Synset('deposit.v.02'), Synset('bank.v.07'), Synset('trust.v.01')]
bank has 18 definitions


Each of these synsets is a different meaning/usage of bank. Let's break these down. 
* Each synset is labeled by `lemma.POS.id`
* `lemma` is the root word this synset is named after
* `POS` is the **p**art **o**f **s**peech (verb, noun, adjective, or adverb) this definition refers to
* `id` is further used to distinguish when `lemma` and `POS` are identical 
    * such as when there are multiple meanings of the noun `bank`

In [8]:
for synset in bank_synsets:
    print(str(synset).ljust(32), synset.definition())
    # print(synset.examples(), '\n') # uses in a sentence

Synset('bank.n.01')              sloping land (especially the slope beside a body of water)
Synset('depository_financial_institution.n.01') a financial institution that accepts deposits and channels the money into lending activities
Synset('bank.n.03')              a long ridge or pile
Synset('bank.n.04')              an arrangement of similar objects in a row or in tiers
Synset('bank.n.05')              a supply or stock held in reserve for future use (especially in emergencies)
Synset('bank.n.06')              the funds held by a gambling house or the dealer in some gambling games
Synset('bank.n.07')              a slope in the turn of a road or track; the outside is higher than the inside in order to reduce the effects of centrifugal force
Synset('savings_bank.n.02')      a container (usually with a slot in the top) for keeping money at home
Synset('bank.n.09')              a building in which the business of banking transacted
Synset('bank.n.10')              a flight maneuver; air

### Group Exercise
Try some other words. What's the word you can find with the most definitions? The least?

What other interesting things do you notice?

Look at the documentation here: [https://www.nltk.org/api/nltk.corpus.reader.wordnet.html#nltk.corpus.reader.wordnet.Synset](https://www.nltk.org/api/nltk.corpus.reader.wordnet.html#nltk.corpus.reader.wordnet.Synset)

In [8]:
# explore a little bit, and note anything interesting you find

## Lemmas
WordNet only considers the base form of words - the word without any inflections (things that modify tense, plurality, case, etc.). 

So, WordNet thinks of all of these as "jump" - they share synsets:
* `jump`
* `jumped`
* `jumps`
* `JUMP`
* `jumping`
* and probably more...

Let's look at the lemmas for the first definition of jump that's a noun. 

In [6]:
jump_synset = wn.synset("jump.n.01")

print(jump_synset.name(), '-', jump_synset.definition())
# the last entry in each lemma is the base word that lemma refers to
print(jump_synset.lemmas())

jump.n.01 - a sudden and decisive increase
[Lemma('jump.n.01.jump'), Lemma('jump.n.01.leap')]


## Semantic Relations
Remember, we're building towards evaluating semantic similarity here - we want to be able to compare how similar the meanings of two words are. To help us do that, WordNet keeps track of lots of different kinds of relationships between words. 

### Group Exercise
Discuss for a few minutes. If you were trying to evaluate how similar the meanings of two words were, what relationships/information would you want about them?
* ...

### Synonymy
Two words are synonyms if they have the same or similar meaning usage. Think about our "sofa" and "couch" example from earlier. 

We get this one for free. WordNet is structured around the idea of **syn**onym sets, so to get the synonyms of a word, we just look at the other words in its synsets.  

In [11]:
sofa_synset = wn.synset("sofa.n.01")
print(sofa_synset)
for lemma in sofa_synset.lemmas():
    print(lemma.name())

# WordNet also provides a shorthand for this
wn.synonyms("sofa")

Synset('sofa.n.01')
sofa
couch
lounge


[['couch', 'lounge']]

### Antonymy
Two words are antonyms if they have opposite meanings, such as "hot" and "cold". This is defined on a per-lemma basis, and it's not entirely clear why this is - it probably has something to do with the fact that a Lemma in WordNet is defined at a more specific level than the synset it's in.


In [12]:
# unsurprisingly, "hot" has lots of definitions, but we'll just use the first adjective definition here
hot_lemma = wn.lemma("hot.a.01.hot")
print(hot_lemma.antonyms())

# this will error
hot_lemma.synset().antonyms()

[Lemma('cold.a.01.cold')]


AttributeError: 'Synset' object has no attribute 'antonyms'

### Hypernyms and Hyponyms
These relations have to do with the generality of synsets. 
* **hypernym**: a more *general* (higher-level) concept
* **hyponym**: a more *specific* (lower-level) concept

This can also be thought of as an "is-a" relationship: a border collie *is a* dog, so we'd say that "border collie" is a hyp*o*nym of "dog", and "dog" is a hyp*er*nym of "border collie".  

<img src="./images/aayla.jpg" width=50%>

*my adorable dog, Aayla*

In [13]:
border_collie = wn.synsets("border_collie")[0]
border_collie.hypernyms() # list of synsets

[Synset('shepherd_dog.n.01')]

In reality, there are a few levels of specificity between "border collie" and "dog" - but we eventually get there. 

In [14]:
synset = wn.synsets("border_collie")[0]
print(synset.name())

while True:
    # just grab the first hypernym
    synset = synset.hypernyms()[0]
    print("↳", synset.name())

    if "dog.n.01" == synset.name():
        break

# you can find hyponyms in the same way; it's Synset.hyponyms()

border_collie.n.01
↳ shepherd_dog.n.01
↳ working_dog.n.01
↳ dog.n.01


The most important takeaway here is the tree-like structure that this creates between words (we'll come back to this later). 

I tried (unsuccessfully) to get an AI model to generate a diagram of this. Back to the whiteboard, I guess. 

### Group Exercise
Pick a few words, traverse their hypernyms until you reach the top, and note where you end up. Did you find any common ancestors? Was there anything unexpected?

The documentation might be helpful here: [https://www.nltk.org/api/nltk.corpus.reader.wordnet.html#nltk.corpus.reader.wordnet.Synset](https://www.nltk.org/api/nltk.corpus.reader.wordnet.html#nltk.corpus.reader.wordnet.Synset)

### Other Relations
* **meronym**: some part of a larger whole
* **holonym**: a whole thing that has parts
* **entailment**: a kind of requirement
    * to snore requires that you're sleeping
* **troponym**: a verb that describes enacting a specific kind of another verb
    * this is just a hyponym on verbs
* ...and more! 

In [15]:
# meronym/holonym
tree = wn.synset('tree.n.01')
print("parts (meronyms) of a tree:")
print(tree.part_meronyms())

stump = wn.synset('stump.n.01')
print("a stump is a part of a tree:")
print(stump.part_holonyms(), '\n')

# entailment
snore = wn.synset('snore.v.01')
print("if you're snoring, then you must be:")
print(snore.entailments(), '\n')

# troponym
run = wn.synset('run.v.01')
print("to do any of the following is to run")
print(run.hyponyms(), '\n')

parts (meronyms) of a tree:
[Synset('burl.n.02'), Synset('limb.n.02'), Synset('trunk.n.01'), Synset('stump.n.01'), Synset('crown.n.07')]
a stump is a part of a tree:
[Synset('tree.n.01')] 

if you're snoring, then you must be:
[Synset('sleep.v.01')] 

to do any of the following is to run
[Synset('rush.v.05'), Synset('run_bases.v.01'), Synset('streak.v.02'), Synset('jog.v.03'), Synset('trot.v.01'), Synset('scurry.v.01'), Synset('sprint.v.01'), Synset('lope.v.01'), Synset('outrun.v.01'), Synset('hare.v.01'), Synset('romp.v.02'), Synset('run.v.33')] 



## Semantic Similarity
This is the main thing we've been building towards - we want to evaluate how similar the *meanings* (semantic value) of two words are. Let's start with the hypernym/hyponym heirarchy we talked about earlier. 

Notice that similar words (like dog and cat) appear more closely in the tree than very different words (like dog and couch). 

This exactly the idea behind the first semantic similarty algorithm - `path_similarity` - get the distance between two words (in the hypernym/hyponym tree) and scale that between 0 and 1
* value of 0 means the words are completely unrelated
* value of 1 means the words are the same

More formally:

${path\_similarity} = \frac{1}{dist + 1}$

where `dist` is the length of the shortest path between the two words. 

In [16]:
dog = wn.synset('dog.n.01')
wolf = wn.synset('wolf.n.01')
canine = wn.synset('canine.n.02')
parrot = wn.synset('parrot.n.01')
cat = wn.synset('cat.n.01')
chicken = wn.synset('chicken.n.01')
astronaut = wn.synset('astronaut.n.01')

print("dog-wolf:", dog.path_similarity(wolf))
print("dog-canine:", dog.path_similarity(canine))
print("dog-parrot:", dog.path_similarity(parrot))
print("dog-cat:", dog.path_similarity(cat))

print("chicken-astronaut:", chicken.path_similarity(astronaut))
print("chicken-chicken:", chicken.path_similarity(chicken))

dog-wolf: 0.3333333333333333
dog-canine: 0.5
dog-parrot: 0.14285714285714285
dog-cat: 0.2
chicken-astronaut: 0.08333333333333333
chicken-chicken: 1.0


Notice something strange here - `poodle` and `corgi` are just as similar as `wall` and `snake`. That doesn't seem right. 

In [17]:
obj = wn.synsets('object')[0]
obj.hyponyms() # includes wall and snake

wall = wn.synset("wall.n.02")
snake = wn.synset("snake.n.05")
print(f"wall and snake have a similarity of {wall.path_similarity(snake)}")

dog = wn.synsets('dog')[0]
dog.hyponyms() # poodle and corgi are both types of dogs

poodle = wn.synset("poodle.n.01")
corgi = wn.synset("corgi.n.01")
print(f"poodle and corgi also have a similarity of {poodle.path_similarity(corgi)}")

wall and snake have a similarity of 0.3333333333333333
poodle and corgi also have a similarity of 0.3333333333333333


### Group Exercise
Why does this happen? What can we do about it? Discuss and write a few ideas.

Don't scroll down. That's cheating. 

In [19]:
for _ in range(25):
    print()
# I *told* you not to scroll down!

This happens because:
* `poodle` and `corgi` are both hyponyms of `dog`
* `wall` and `snake` are both hyponyms of `object`
* thus, the distance between the items in each pair is the same (2)

Let's introduce a new algorithm for similarity - Wu-Palmer similarity. Instead of just using the distance between each object, let's scale that by how deep their common ancestor is in the tree. 

Specifically,

${wup\_similarity} = \frac{2k}{i + j + 2k}$

where `i` and `j` are the respective distances of each word we're comparing from their deepest common ancestor; and `k` is the depth of that ancestor. 


* `poodle` and `corgi` have a common ancestor of `dog`, which is quite deep in the tree
    * this means that the `wup_similarity` fraction is going to be biased more towards $\frac{2k}{2k}$, which simplifies to 1
* `wall` and `snake` have a common ancestor of `object`, which isn't very deep
    * this means the numerator is much closer to zero; so the words don't have as high of a similarity score

In [20]:
obj = wn.synsets('object')[0]
obj.hyponyms() # includes wall and snake

wall = wn.synset("wall.n.02")
snake = wn.synset("snake.n.05")
print(f"wall and snake have a similarity of {wall.wup_similarity(snake)}")

dog = wn.synsets('dog')[0]
dog.hyponyms() # poodle and corgi are both types of dogs

poodle = wn.synset("poodle.n.01")
corgi = wn.synset("corgi.n.01")
print(f"poodle and corgi also have a similarity of {poodle.wup_similarity(corgi)}")

wall and snake have a similarity of 0.75
poodle and corgi also have a similarity of 0.8666666666666667


### Group Exercise
You're building each of the following tools - would you use WordNet or word embeddings for each of them? Why? How might that system work?
* A tool that groups news headlines by subject
* A system that recommends word synonyms in a writing assistant
* A search engine for a medical encyclopedia
* A system for evaluating whether a user's query is on-topic for a special-use chatbot (like Professor Griff)

## Applied Exploration
Look into [other algorithms](https://wn.readthedocs.io/en/latest/api/wn.similarity.html) for calculating semantic similarity with WordNet. Submit a write-up including the following:
* An explanation of how the algorithm works
* Who created it, and why
* Strengths and weaknesses of the algorithm - which types of comparisons does it accurately protray, and which does it do poorly?
* Show it working on a few examples, and compare the scores to either Wu-Palmer or path similarity
    * Bonus points if you implement it yourself or write your own

The Python NLTK library provides a few other semantic similarity algorithms. You can look into these, or find other ones out there. 
* Leacock-Chodorow Similarity
* Resnik Similarity
* Jiang-Conrath Similarity
* Lin Similarity


## Creative Synthesis
* Make a game that selects two random words and has the player guess the similarity between them using a WordNet algorithm or an embeddidng model
* Make something like the New York Times game Connections that generates groups of words based on similarity, or some other relationship
* Make a tool that performs semantic search on some documents (like the AG news headlines) using WordNet similarity